In [ ]:
import os
import sqlite3
import hashlib
from datetime import datetime
from PIL import Image, ImageDraw
import numpy as np
import matplotlib.pyplot as plt
from azure.cognitiveservices.vision.customvision.prediction import CustomVisionPredictionClient
from msrest.authentication import ApiKeyCredentials

class SinkholeReportSystem:
    def __init__(self, db_name="sinkhole_reports.db"):
        self.db_name = db_name
        self.init_database()
        
        # Azure Custom Vision 설정
        self.prediction_endpoint = os.getenv("PREDICTION_ENDPOINT")
        self.prediction_key = os.getenv("PREDICTION_KEY")
        self.project_id = os.getenv("PROJECT_ID")
        self.model_name = os.getenv("MODEL_NAME")
        
        # 포인트 시스템 설정
        self.point_rules = {
            'successful_detection': 100,    # 성공적인 싱크홀 탐지 시
            'high_confidence': 50,          # 높은 신뢰도(80% 이상) 탐지 시 보너스
            'multiple_detection': 30,       # 한 이미지에서 여러 싱크홀 탐지 시 추가
            'first_report_bonus': 200       # 신규 사용자 첫 신고 보너스
        }
    
    def get_connection(self):
        """데이터베이스 연결 반환"""
        return sqlite3.connect(self.db_name)
    
    def init_database(self):
        """데이터베이스 초기화 및 테이블 생성"""
        conn = self.get_connection()
        cursor = conn.cursor()
        
        # 사용자 테이블 생성
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS users (
                user_id TEXT PRIMARY KEY,
                password TEXT NOT NULL,
                points INTEGER DEFAULT 0,
                total_reports INTEGER DEFAULT 0,
                successful_detections INTEGER DEFAULT 0,
                registration_date TEXT NOT NULL,
                last_login TEXT
            )
        ''')
        
        # 신고 테이블 생성
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS reports (
                report_id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id TEXT NOT NULL,
                timestamp TEXT NOT NULL,
                image_file TEXT NOT NULL,
                location TEXT,
                description TEXT,
                detection_count INTEGER DEFAULT 0,
                high_confidence_count INTEGER DEFAULT 0,
                earned_points INTEGER DEFAULT 0,
                first_report_bonus BOOLEAN DEFAULT FALSE,
                FOREIGN KEY (user_id) REFERENCES users (user_id)
            )
        ''')
        
        # 탐지 결과 테이블 생성
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS detections (
                detection_id INTEGER PRIMARY KEY AUTOINCREMENT,
                report_id INTEGER NOT NULL,
                prediction_id INTEGER NOT NULL,
                tag_name TEXT NOT NULL,
                confidence REAL NOT NULL,
                bbox_left REAL NOT NULL,
                bbox_top REAL NOT NULL,
                bbox_width REAL NOT NULL,
                bbox_height REAL NOT NULL,
                FOREIGN KEY (report_id) REFERENCES reports (report_id)
            )
        ''')
        
        conn.commit()
        conn.close()
    
    def hash_password(self, password):
        """패스워드 해시화"""
        return hashlib.sha256(password.encode()).hexdigest()
    
    def register_user(self, user_id, password):
        """새 사용자 등록"""
        conn = self.get_connection()
        cursor = conn.cursor()
        
        try:
            # 중복 사용자 확인
            cursor.execute("SELECT user_id FROM users WHERE user_id = ?", (user_id,))
            if cursor.fetchone():
                return False, "이미 존재하는 사용자 ID입니다."
            
            # 새 사용자 등록
            cursor.execute('''
                INSERT INTO users (user_id, password, registration_date)
                VALUES (?, ?, ?)
            ''', (user_id, self.hash_password(password), datetime.now().isoformat()))
            
            conn.commit()
            return True, "사용자 등록이 완료되었습니다."
            
        except sqlite3.Error as e:
            return False, f"데이터베이스 오류: {str(e)}"
        finally:
            conn.close()
    
    def login_user(self, user_id, password):
        """사용자 로그인"""
        conn = self.get_connection()
        cursor = conn.cursor()
        
        try:
            # 사용자 정보 확인
            cursor.execute("SELECT password FROM users WHERE user_id = ?", (user_id,))
            result = cursor.fetchone()
            
            if not result:
                return False, "존재하지 않는 사용자 ID입니다."
            
            hashed_password = self.hash_password(password)
            if result[0] != hashed_password:
                return False, "비밀번호가 일치하지 않습니다."
            
            # 마지막 로그인 시간 업데이트
            cursor.execute('''
                UPDATE users SET last_login = ? WHERE user_id = ?
            ''', (datetime.now().isoformat(), user_id))
            
            conn.commit()
            return True, "로그인 성공"
            
        except sqlite3.Error as e:
            return False, f"데이터베이스 오류: {str(e)}"
        finally:
            conn.close()
    
    def get_user_info(self, user_id):
        """사용자 정보 조회"""
        conn = self.get_connection()
        cursor = conn.cursor()
        
        try:
            cursor.execute('''
                SELECT user_id, points, total_reports, successful_detections, 
                       registration_date, last_login
                FROM users WHERE user_id = ?
            ''', (user_id,))
            
            result = cursor.fetchone()
            if result:
                return {
                    'user_id': result[0],
                    'points': result[1],
                    'total_reports': result[2],
                    'successful_detections': result[3],
                    'registration_date': result[4],
                    'last_login': result[5]
                }
            return None
            
        except sqlite3.Error as e:
            print(f"데이터베이스 오류: {str(e)}")
            return None
        finally:
            conn.close()
    
    def calculate_points(self, detection_results):
        """탐지 결과를 바탕으로 포인트 계산"""
        total_points = 0
        detection_count = 0
        high_confidence_count = 0
        
        for prediction in detection_results.predictions:
            confidence = prediction.probability * 100
            
            if confidence > 50:  # 탐지 성공
                detection_count += 1
                total_points += self.point_rules['successful_detection']
                
                if confidence > 80:  # 고신뢰도 보너스
                    high_confidence_count += 1
                    total_points += self.point_rules['high_confidence']
        
        # 다중 탐지 보너스
        if detection_count > 1:
            total_points += self.point_rules['multiple_detection'] * (detection_count - 1)
        
        return total_points, detection_count, high_confidence_count
    
    def add_points(self, user_id, points, detection_count):
        """사용자에게 포인트 추가"""
        conn = self.get_connection()
        cursor = conn.cursor()
        
        try:
            # 현재 사용자 정보 조회
            cursor.execute('''
                SELECT points, total_reports, successful_detections 
                FROM users WHERE user_id = ?
            ''', (user_id,))
            
            result = cursor.fetchone()
            if not result:
                return False, False
            
            current_points, total_reports, successful_detections = result
            
            # 첫 신고 보너스 체크
            first_report_bonus = False
            if total_reports == 0:
                points += self.point_rules['first_report_bonus']
                first_report_bonus = True
            
            # 사용자 정보 업데이트
            cursor.execute('''
                UPDATE users 
                SET points = ?, total_reports = ?, successful_detections = ?
                WHERE user_id = ?
            ''', (current_points + points, total_reports + 1, 
                  successful_detections + detection_count, user_id))
            
            conn.commit()
            return True, first_report_bonus
            
        except sqlite3.Error as e:
            print(f"데이터베이스 오류: {str(e)}")
            return False, False
        finally:
            conn.close()
    
    def save_report(self, user_id, image_file, location, description, 
                   detection_count, high_confidence_count, earned_points, 
                   first_report_bonus, detection_results):
        """신고 데이터를 데이터베이스에 저장"""
        conn = self.get_connection()
        cursor = conn.cursor()
        
        try:
            # 신고 정보 저장
            cursor.execute('''
                INSERT INTO reports (user_id, timestamp, image_file, location, 
                                   description, detection_count, high_confidence_count, 
                                   earned_points, first_report_bonus)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (user_id, datetime.now().isoformat(), image_file, location, 
                  description, detection_count, high_confidence_count, 
                  earned_points, first_report_bonus))
            
            report_id = cursor.lastrowid
            
            # 탐지 결과 저장
            for i, prediction in enumerate(detection_results.predictions):
                if (prediction.probability * 100) > 50:
                    cursor.execute('''
                        INSERT INTO detections (report_id, prediction_id, tag_name, 
                                              confidence, bbox_left, bbox_top, 
                                              bbox_width, bbox_height)
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                    ''', (report_id, i, prediction.tag_name, 
                          prediction.probability * 100,
                          prediction.bounding_box.left,
                          prediction.bounding_box.top,
                          prediction.bounding_box.width,
                          prediction.bounding_box.height))
            
            conn.commit()
            return report_id
            
        except sqlite3.Error as e:
            print(f"데이터베이스 오류: {str(e)}")
            return None
        finally:
            conn.close()
    
    def process_sinkhole_image(self, user_id, image_file, location=None, description=None):
        """싱크홀 이미지 처리 및 포인트 부여"""
        try:
            # Azure Custom Vision 설정
            credentials = ApiKeyCredentials(in_headers={"Prediction-key": self.prediction_key})
            predictor = CustomVisionPredictionClient(endpoint=self.prediction_endpoint, credentials=credentials)
            
            # 이미지 로드
            image = Image.open(image_file)
            
            # 객체 탐지 수행
            with open(image_file, mode='rb') as image_data:
                results = predictor.detect_image(self.project_id, self.model_name, image_data)
            
            # 포인트 계산
            earned_points, detection_count, high_confidence_count = self.calculate_points(results)
            
            # 포인트 추가 및 첫 신고 보너스 확인
            success, first_report_bonus = self.add_points(user_id, earned_points, detection_count)
            
            if not success:
                return {'success': False, 'error': '사용자 정보 업데이트 실패'}
            
            # 신고 데이터 저장
            report_id = self.save_report(user_id, image_file, location, description,
                                       detection_count, high_confidence_count, 
                                       earned_points, first_report_bonus, results)
            
            if not report_id:
                return {'success': False, 'error': '신고 데이터 저장 실패'}
            
            # 시각화 생성
            output_file = self.create_detection_visualization(image, results, report_id, 
                                                            earned_points, detection_count)
            
            # 현재 총 포인트 조회
            user_info = self.get_user_info(user_id)
            total_points = user_info['points'] if user_info else 0
            
            return {
                'success': True,
                'earned_points': earned_points,
                'detection_count': detection_count,
                'high_confidence_count': high_confidence_count,
                'total_points': total_points,
                'output_file': output_file,
                'report_id': report_id,
                'first_report_bonus': first_report_bonus
            }
            
        except Exception as e:
            return {
                'success': False,
                'error': str(e)
            }
    
    def create_detection_visualization(self, image, results, report_id, earned_points, detection_count):
        """탐지 결과 시각화 생성"""
        fig = plt.figure(figsize=(12, 8))
        plt.axis('off')
        
        # 이미지 그리기
        draw = ImageDraw.Draw(image)
        h, w = np.array(image).shape[:2]
        lineWidth = max(int(w/100), 2)
        color = 'magenta'
        
        for prediction in results.predictions:
            if (prediction.probability * 100) > 50:
                left = prediction.bounding_box.left * w
                top = prediction.bounding_box.top * h
                width = prediction.bounding_box.width * w
                height = prediction.bounding_box.height * h
                
                # 박스 그리기
                points = ((left, top), (left+width, top), (left+width, top+height), (left, top+height), (left, top))
                draw.line(points, fill=color, width=lineWidth)
                
                # 레이블 추가
                label = f"{prediction.tag_name} {prediction.probability*100:.1f}%"
                plt.annotate(label, (left, top-20), color=color, fontsize=10, weight='bold')
        
        plt.imshow(image)
        
        # 제목에 포인트 정보 추가
        title = f"탐지 결과 - 신고 ID: {report_id}\n"
        title += f"탐지된 싱크홀: {detection_count}개 | 획득 포인트: {earned_points}점"
        plt.title(title, fontsize=14, weight='bold', pad=20)
        
        # 파일 저장
        output_file = f"report_{report_id}.jpg"
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        plt.close()
        
        return output_file
    
    def get_user_reports(self, user_id, limit=None):
        """사용자의 신고 내역 조회"""
        conn = self.get_connection()
        cursor = conn.cursor()
        
        try:
            query = '''
                SELECT report_id, timestamp, image_file, location, description,
                       detection_count, high_confidence_count, earned_points,
                       first_report_bonus
                FROM reports 
                WHERE user_id = ?
                ORDER BY timestamp DESC
            '''
            
            if limit:
                query += f' LIMIT {limit}'
            
            cursor.execute(query, (user_id,))
            results = cursor.fetchall()
            
            reports = []
            for row in results:
                reports.append({
                    'report_id': row[0],
                    'timestamp': row[1],
                    'image_file': row[2],
                    'location': row[3],
                    'description': row[4],
                    'detection_count': row[5],
                    'high_confidence_count': row[6],
                    'earned_points': row[7],
                    'first_report_bonus': bool(row[8])
                })
            
            return reports
            
        except sqlite3.Error as e:
            print(f"데이터베이스 오류: {str(e)}")
            return []
        finally:
            conn.close()
    
    def get_leaderboard(self, top_n=10):
        """포인트 상위 랭킹 조회"""
        conn = self.get_connection()
        cursor = conn.cursor()
        
        try:
            cursor.execute('''
                SELECT user_id, points, total_reports, successful_detections
                FROM users
                ORDER BY points DESC
                LIMIT ?
            ''', (top_n,))
            
            results = cursor.fetchall()
            leaderboard = []
            
            for i, row in enumerate(results):
                leaderboard.append({
                    'rank': i + 1,
                    'user_id': row[0],
                    'points': row[1],
                    'total_reports': row[2],
                    'successful_detections': row[3]
                })
            
            return leaderboard
            
        except sqlite3.Error as e:
            print(f"데이터베이스 오류: {str(e)}")
            return []
        finally:
            conn.close()
    
    def get_report_details(self, report_id):
        """특정 신고의 상세 정보 조회"""
        conn = self.get_connection()
        cursor = conn.cursor()
        
        try:
            # 신고 기본 정보 조회
            cursor.execute('''
                SELECT r.report_id, r.user_id, r.timestamp, r.image_file, 
                       r.location, r.description, r.detection_count, 
                       r.high_confidence_count, r.earned_points, r.first_report_bonus
                FROM reports r
                WHERE r.report_id = ?
            ''', (report_id,))
            
            report_result = cursor.fetchone()
            if not report_result:
                return None
            
            # 탐지 결과 조회
            cursor.execute('''
                SELECT prediction_id, tag_name, confidence, bbox_left, 
                       bbox_top, bbox_width, bbox_height
                FROM detections
                WHERE report_id = ?
                ORDER BY prediction_id
            ''', (report_id,))
            
            detection_results = cursor.fetchall()
            
            report_details = {
                'report_id': report_result[0],
                'user_id': report_result[1],
                'timestamp': report_result[2],
                'image_file': report_result[3],
                'location': report_result[4],
                'description': report_result[5],
                'detection_count': report_result[6],
                'high_confidence_count': report_result[7],
                'earned_points': report_result[8],
                'first_report_bonus': bool(report_result[9]),
                'detections': []
            }
            
            for detection in detection_results:
                report_details['detections'].append({
                    'prediction_id': detection[0],
                    'tag_name': detection[1],
                    'confidence': detection[2],
                    'bounding_box': {
                        'left': detection[3],
                        'top': detection[4],
                        'width': detection[5],
                        'height': detection[6]
                    }
                })
            
            return report_details
            
        except sqlite3.Error as e:
            print(f"데이터베이스 오류: {str(e)}")
            return None
        finally:
            conn.close()


# 사용 예시 및 메인 함수
def main():
    # 시스템 초기화
    system = SinkholeReportSystem()
    current_user = None
    
    print("=== 싱크홀 신고 시스템 (Database Version) ===")
    
    while True:
        print("\n1. 사용자 등록")
        print("2. 로그인")
        print("3. 싱크홀 신고")
        print("4. 내 정보 조회")
        print("5. 신고 내역 조회")
        print("6. 랭킹 조회")
        print("7. 신고 상세 조회")
        print("8. 종료")
        
        choice = input("\n선택하세요: ")
        
        if choice == '1':
            user_id = input("사용자 ID: ")
            password = input("비밀번호: ")
            success, message = system.register_user(user_id, password)
            print(message)
        
        elif choice == '2':
            user_id = input("사용자 ID: ")
            password = input("비밀번호: ")
            success, message = system.login_user(user_id, password)
            print(message)
            
            if success:
                current_user = user_id
                print(f"현재 로그인 사용자: {current_user}")
        
        elif choice == '3':
            if not current_user:
                print("먼저 로그인하세요.")
                continue
            
            image_file = input("이미지 파일 경로: ")
            if not os.path.exists(image_file):
                print("파일이 존재하지 않습니다.")
                continue
                
            location = input("위치 (선택사항): ") or None
            description = input("설명 (선택사항): ") or None
            
            result = system.process_sinkhole_image(current_user, image_file, location, description)
            
            if result['success']:
                print(f"\n신고 완료!")
                print(f"신고 ID: {result['report_id']}")
                print(f"탐지된 싱크홀: {result['detection_count']}개")
                print(f"획득 포인트: {result['earned_points']}점")
                if result['first_report_bonus']:
                    print("🎉 첫 신고 보너스 200점 추가!")
                print(f"총 포인트: {result['total_points']}점")
                print(f"결과 이미지: {result['output_file']}")
            else:
                print(f"오류 발생: {result['error']}")
        
        elif choice == '4':
            if not current_user:
                print("먼저 로그인하세요.")
                continue
            
            user_info = system.get_user_info(current_user)
            if user_info:
                print(f"\n=== {current_user} 정보 ===")
                print(f"포인트: {user_info['points']}점")
                print(f"총 신고 수: {user_info['total_reports']}회")
                print(f"성공적인 탐지: {user_info['successful_detections']}회")
                print(f"가입일: {user_info['registration_date']}")
                if user_info['last_login']:
                    print(f"마지막 로그인: {user_info['last_login']}")
        
        elif choice == '5':
            if not current_user:
                print("먼저 로그인하세요.")
                continue
            
            reports = system.get_user_reports(current_user, limit=5)
            print(f"\n=== {current_user} 신고 내역 (최근 5개) ===")
            
            if not reports:
                print("신고 내역이 없습니다.")
            else:
                for report in reports:
                    print(f"신고 ID: {report['report_id']}")
                    print(f"일시: {report['timestamp']}")
                    print(f"위치: {report.get('location', '미지정')}")
                    print(f"탐지 수: {report['detection_count']}개")
                    print(f"획득 포인트: {report['earned_points']}점")
                    if report['first_report_bonus']:
                        print("🎉 첫 신고 보너스 포함")
                    print("-" * 40)
        
        elif choice == '6':
            leaderboard = system.get_leaderboard()
            print("\n=== 포인트 랭킹 ===")
            if not leaderboard:
                print("등록된 사용자가 없습니다.")
            else:
                for user in leaderboard:
                    print(f"{user['rank']}위: {user['user_id']} - {user['points']}점 "
                          f"(신고 {user['total_reports']}회, 탐지 {user['successful_detections']}회)")
        
        elif choice == '7':
            report_id = input("신고 ID: ")
            try:
                report_id = int(report_id)
                details = system.get_report_details(report_id)
                
                if details:
                    print(f"\n=== 신고 상세 정보 (ID: {report_id}) ===")
                    print(f"신고자: {details['user_id']}")
                    print(f"일시: {details['timestamp']}")
                    print(f"이미지: {details['image_file']}")
                    print(f"위치: {details.get('location', '미지정')}")
                    print(f"설명: {details.get('description', '없음')}")
                    print(f"탐지 수: {details['detection_count']}개")
                    print(f"고신뢰도 탐지: {details['high_confidence_count']}개")
                    print(f"획득 포인트: {details['earned_points']}점")
                    
                    if details['detections']:
                        print("\n탐지 결과:")
                        for detection in details['detections']:
                            print(f"  - {detection['tag_name']}: {detection['confidence']:.1f}%")
                            bbox = detection['bounding_box']
                            print(f"    위치: ({bbox['left']:.3f}, {bbox['top']:.3f}) "
                                  f"크기: {bbox['width']:.3f} x {bbox['height']:.3f}")
                else:
                    print("해당 신고를 찾을 수 없습니다.")
            except ValueError:
                print("올바른 신고 ID를 입력하세요.")
        
        elif choice == '8':
            print("프로그램을 종료합니다.")
            break
        
        else:
            print("올바른 번호를 선택하세요.")

if __name__ == "__main__":
    main()
    